In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

from multiprocessing import Pool, cpu_count

import dask
import dask.dataframe as dd
from dask.distributed import Client
client = Client(n_workers=20, memory_limit="10GB", interface='lo')

from pprint import pprint

In [2]:
augmented_data_path = "../data/augmented_us-counties-states_latest.csv"
augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True))

In [3]:
# Define a function to calculate RMSE and MAE
def calc_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

In [4]:
# Define the window size range
window_size_range = range(2, 15)

# Initialize variables to store the best window size and its corresponding RMSE and MAE
best_window_size_rmse = {}
best_window_size_mae = {}

In [5]:
# Loop through the time points
for time_point in range(len(augmented_df["days_from_start"]) - 7):
    print(f"Processing time point {time_point}...")

    # Define the training and validation data for this time point
    train_data = augmented_df.iloc[time_point : time_point + 7,:]
    val_data = augmented_df.iloc[time_point + 7 : time_point + 14,:]

    # Initialize variables to store the RMSE and MAE for each window size
    rmse_scores = {}
    mae_scores = {}

    # Loop through the window sizes
    for window_size in window_size_range:
        print(f"Processing window size {window_size}...")

        # Define the training and validation data for this window size
        train_X = train_data.iloc[-window_size - 7 : -7][["days_from_start"]]
        train_y = np.log(train_data.iloc[-window_size - 7 : -7]["rolled_cases"])
        val_X = val_data[["days_from_start"]]
        val_y = np.log(val_data["rolled_cases"])

        # Fit a linear regression model to the training data
        lr = LinearRegression()
        lr.fit(train_X, train_y)

        # Make predictions on the validation data
        val_pred = lr.predict(val_X)

        # Calculate the RMSE and MAE for this window size
        rmse, mae = calc_metrics(val_y, val_pred)
        rmse_scores[window_size] = rmse
        mae_scores[window_size] = mae

    # Find the window size with the lowest RMSE and MAE
    best_window_size_rmse[time_point] = min(rmse_scores, key=rmse_scores.get)
    best_window_size_mae[time_point] = min(mae_scores, key=mae_scores.get)

    print(f"Best window size based on RMSE: {best_window_size_rmse[time_point]}")
    print(f"Best window size based on MAE: {best_window_size_mae[time_point]}")

Processing time point 0...


NotImplementedError: 'DataFrame.iloc' only supports selecting columns. It must be used like 'df.iloc[:, column_indexer]'.

distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
